# ResNet18 Data Source Screening

This notebook trains one ImageNet-pretrained ResNet18 classifier on the labeled real fixation ERP images and applies it to every standardized Week-19 data source bundle that is available locally.

The goal is not to make an automatic include/exclude decision. The model is used as a screening tool: for each data source and each available sort variable, the notebook ranks the ERP images by `P(class)` and plots the top candidates directly in the notebook. A human reviewer can then decide which data sources should be included.

In [ ]:
using InteractiveUtils
versioninfo()

## Setup

The helper module reuses the training and preprocessing code from `resnet_fixation_generalization_experiment.jl`. The image convention is the same as in the previous ResNet notebook: trials are rows, time is columns, trials are sorted by the selected sort variable using the Week-19 preview tie-breakers, values are z-scored, smoothed with the shared Gaussian reference pipeline, and resized to `64 x 64`.

No candidate images are written to disk. Plotting functions return `CairoMakie` figures for notebook display only.

In [ ]:
import Pkg

ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_20")
Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using DataFrames

include(joinpath(NOTEBOOK_DIR, "resnet18_data_source_screening.jl"))
using .Week20ResNet18DataSourceScreening

println("Notebook directory: ", NOTEBOOK_DIR)
println("Week-19 data-source notebooks: ", Week20ResNet18DataSourceScreening.WEEK19_DATA_SOURCE_NOTEBOOK_DIR)

## Discover Week-19 Data Sources

The discovery step reads the Week-19 data-source notebooks, extracts their `DATASET_KEY`, checks whether a standardized bundle exists under `notebooks/datasets`, and records whether the original preview notebook used baseline correction.

Rows with `ready == false` are kept in the status table so excluded notebooks such as auxiliary downloads or missing/login-gated sources remain visible.

In [ ]:
source_status_df = discover_week19_data_sources()
source_status_df

In [ ]:
source_status_df[source_status_df.ready .== false, [:notebook_file, :dataset_key, :skip_reason]]

## Run Configuration

`DATASET_KEYS = nothing` means all ready Week-19 data-source bundles are screened. `MAX_CHANNELS = nothing` means every channel is used; set it to a small integer only for a quick smoke test.

In [ ]:
SCREENING_EPOCHS = Week20ResNet18DataSourceScreening.Generalization.TRAIN_EPOCHS
TOP_N = Week20ResNet18DataSourceScreening.TOP_N_CANDIDATES
DATASET_KEYS = nothing
MAX_CHANNELS = nothing

(SCREENING_EPOCHS = SCREENING_EPOCHS, TOP_N = TOP_N, DATASET_KEYS = DATASET_KEYS, MAX_CHANNELS = MAX_CHANNELS)

## Train ResNet18

The training policy matches the previous ResNet experiment: labeled `class` rows keep all four modulo-4 parts, labeled `no_class` rows are split modulo-4 but only one seeded part is kept to reduce class imbalance.

In [ ]:
training = train_resnet18_screening_model(nepochs = SCREENING_EPOCHS)
training.train_metrics_df

In [ ]:
training.history_df

## Materialize ERP Images For Screening

For each ready data source, the notebook creates one ERP image for every channel and every available sort variable. Multi-subject sources are merged across subjects, matching the Week-19 preview convention.

In [ ]:
target_df = materialize_all_source_images(
    source_status_df;
    dataset_keys = DATASET_KEYS,
    max_channels = MAX_CHANNELS,
)

target_metadata_df = select(target_df, Not(:processed_img))
first(target_metadata_df, min(10, nrow(target_metadata_df)))

In [ ]:
combine(groupby(target_metadata_df, [:dataset_key, :sort_col]), nrow => :n_images)

## Classify All Source Images

The classifier output is interpreted as a screening score. `prob_class` is the main ranking value for candidate inspection.

In [ ]:
prediction_df = predict_source_images(
    training.model,
    target_df;
    device = training.device,
)

dataset_sort_summary_df = summarize_by_dataset_sort(prediction_df)
dataset_summary_df = summarize_by_dataset(prediction_df)
candidates_df = top_class_candidates(prediction_df; n = TOP_N)
candidate_images_df = attach_candidate_images(candidates_df, target_df)

dataset_summary_df

In [ ]:
dataset_sort_summary_df

In [ ]:
candidates_df

## Screening Overview

This plot summarizes the strongest `P(class)` signal per data source. It is a prioritization view only; source inclusion remains a human decision based on the candidate images below.

In [ ]:
plot_screening_overview(dataset_summary_df)

## Candidate Plots

Each figure corresponds to one data source. Rows are sort variables and columns are the top `TOP_N` candidates ranked by `P(class)`. If a candidate is still predicted as `no_class`, the title makes that visible; this means it is only among the best available candidates for that sort variable, not necessarily a strong positive example.

The figures are displayed in the notebook and are not saved as image files.

In [ ]:
for dataset_key in String.(dataset_summary_df.dataset_key)
    display(plot_dataset_candidate_grid(candidate_images_df, dataset_key; n = TOP_N))
end

## Single Source Re-Inspection

Use this cell to re-display one source after inspecting the summary tables.

In [ ]:
DATASET_KEY_TO_INSPECT = String(dataset_summary_df.dataset_key[1])
plot_dataset_candidate_grid(candidate_images_df, DATASET_KEY_TO_INSPECT; n = TOP_N)

## Interpretation Notes

Use `dataset_summary_df` to find data sources with strong overall evidence. Use `dataset_sort_summary_df` to see which sort variables create the strongest candidate images. Use `candidates_df` for exact channel, probability, and trial-count metadata.

A high `P(class)` candidate should be treated as an inspection priority, not proof that the whole source should be included. The final include/exclude decision should be made by reviewing the plotted ERP image patterns and the source metadata together.